# EA3 — Comparación de datos climáticos: Medellín vs. Madrid

## A. Planteamiento de la comparación
* **Pregunta guía:** ¿Qué similitudes y diferencias se observan en la temperatura máxima diaria y precipitación entre Medellín (Colombia) y Madrid (España) durante el año 2023?
* **Predicción:** Esperamos que Medellín muestre una temperatura máxima muy estable durante todo el año debido a su ubicación tropical, mientras que Madrid exhibirá una fuerte estacionalidad (veranos muy calurosos e inviernos fríos).
* **Periodo común:** 1 de enero de 2023 al 31 de diciembre de 2023 (Frecuencia diaria).
* **Variables:** Temperatura máxima diaria (°C) y Precipitación total diaria (mm).
* **Fuentes:** Open-Meteo Historical Weather API (Datos derivados de modelos de reanálisis ERA5).

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import requests

# B. Carga y revisión de los datos (Descarga directa desde la API para garantizar simetría)
def cargar_clima(lat, lon, timezone):
    url = f"https://archive-api.open-meteo.com/v1/archive?latitude={lat}&longitude={lon}&start_date=2023-01-01&end_date=2023-12-31&daily=temperature_2m_max,precipitation_sum&timezone={timezone}"
    res = requests.get(url).json()
    df = pd.DataFrame(res['daily'])
    df['time'] = pd.to_datetime(df['time'])
    df.rename(columns={'time': 'fecha', 'temperature_2m_max': 'temp_max', 'precipitation_sum': 'precipitacion'}, inplace=True)
    return df

# Cargar datos de Medellín y Madrid
df_medellin = cargar_clima(6.2518, -75.5636, "America%2FBogota")
df_madrid = cargar_clima(40.4165, -3.7026, "Europe%2FMadrid")

print("--- REVISIÓN MEDELLÍN ---")
print(f"Dimensiones: {df_medellin.shape}")
print(f"Nulos:\n{df_medellin.isna().sum()}")
print(f"Duplicados: {df_medellin.duplicated().sum()}")

print("\n--- REVISIÓN MADRID ---")
print(f"Dimensiones: {df_madrid.shape}")
print(f"Nulos:\n{df_madrid.isna().sum()}")

display(df_medellin.head(3))

In [ ]:
# C. Análisis con NumPy (Arrays, Broadcasting y Máscaras)
# 1. Extracción a arrays de NumPy
temp_med = df_medellin['temp_max'].to_numpy()
precip_med = df_medellin['precipitacion'].to_numpy()

temp_mad = df_madrid['temp_max'].to_numpy()
precip_mad = df_madrid['precipitacion'].to_numpy()

# 2. Broadcasting: Calcular la diferencia de temperatura de Madrid respecto a la media de Medellín
media_temp_med = np.mean(temp_med)
diferencia_madrid_vs_medellin = temp_mad - media_temp_med # Broadcasting escalar

# 3. Cortes (Slicing): Inspección de la primera semana de enero
print(f"Temperaturas 1ra semana Medellín: {temp_med[:7]}")
print(f"Temperaturas 1ra semana Madrid: {temp_mad[:7]}")

# 4. Máscaras booleanas (Umbral definido: 28°C para días calurosos, > 0 para lluvia)
calor_med = temp_med > 28
lluvia_med = precip_med > 0

calor_mad = temp_mad > 28
lluvia_mad = precip_mad > 0

# 5. Estadísticas descriptivas
print("\n--- ESTADÍSTICAS COMPARATIVAS 2023 ---")
print(f"MEDELLÍN -> Temp Promedio: {np.mean(temp_med):.1f}°C | Temp Max: {np.max(temp_med):.1f}°C | Días lluviosos: {np.sum(lluvia_med)} | Precip total: {np.sum(precip_med):.1f} mm")
print(f"MADRID   -> Temp Promedio: {np.mean(temp_mad):.1f}°C | Temp Max: {np.max(temp_mad):.1f}°C | Días lluviosos: {np.sum(lluvia_mad)} | Precip total: {np.sum(precip_mad):.1f} mm")